# Table 1: DNSSEC and NSEC/3 adoption
Compute Table 1 in *Dissecting NSEC and NSEC3 Deployment in the Wild* (April 6, 2026 snapshot).

**Default mode (`summary`)** loads the committed count CSVs in `script/res/` and computes every percentage. This reproduces the table without downloading or probing domains.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Works from the repository root, script/, or this notebook's directory.
REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "script/res/20260406_domcnt_stats.csv").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the dnsnsec26 repository, or set REPO_ROOT explicitly.")

MODE = "summary"  # "summary" or "raw"
DATE = "20260406"
EXPORT = True
OUTPUT_DIR = REPO_ROOT / "output/jupyter-notebook/table1_results"
ZONES = ["com", "net", "org", "umbrella", "tranco", "radar", "majestic"]
LABELS = {"com": ".com", "net": ".net", "org": ".org",
          "umbrella": "Umbrella", "tranco": "Tranco",
          "radar": "Radar", "majestic": "Majestic"}
print(f"Input mode: {MODE}; snapshot: {DATE}")

Input mode: summary; snapshot: 20260406


## Count definitions and input paths
- Source **SOA**: distinct `query_name` with SOA query/response and a non-null `soa_mname`.
- Source **DNSKEY**: distinct `query_name` with DNSKEY query/response and non-null `dnskey_flags`.
- Source **NSEC / NSEC3**: distinct query names in merged `*_ns_everything.parquet`, including the auxiliary scan.
- WWW **DNSKEY**: distinct names in the DNSKEY-enabled apex target list. Popular rankings may contain several FQDNs per apex, so this differs from their source DNSKEY count.
- WWW **NSEC / NSEC3**: distinct `probe_fqdn` with a `basename` in that target list.


In [2]:
def read_summary_counts():
    result_dir = REPO_ROOT / "script/res"
    source = pd.read_csv(result_dir / f"{DATE}_domcnt_stats.csv")
    www = pd.read_csv(result_dir / f"{DATE}_www_domcnt_stats.csv")
    return source, www

if MODE == "summary":
    source_counts, www_counts = read_summary_counts()
else:
    raise ValueError(f"Unknown MODE: {MODE!r}")

## Compute the table
Source DNSSEC adoption = `100 × source DNSKEY / source SOA`.

For each mechanism, source adoption = `100 × source NSEC(or NSEC3) / source DNSKEY`; WWW adoption = `100 × WWW NSEC(or NSEC3) / WWW DNSKEY`.

Percentages are stored as numbers on the 0–100 scale. Rounding to two decimals is applied only for presentation. The table uses apex counts for the three TLD datasets and FQDN counts for the four source popularity rankings.

In [3]:
def validate_counts(frame, count_columns):
    required = {"zone", *count_columns}
    if not required.issubset(frame.columns):
        raise ValueError(f"Missing columns: {required - set(frame.columns)}")
    if frame["zone"].duplicated().any() or set(frame["zone"]) != set(ZONES):
        raise ValueError("Expected one row for each of the seven datasets.")
    ordered = frame.set_index("zone").loc[ZONES, count_columns].copy()
    values = ordered.apply(pd.to_numeric, errors="raise")
    if values.isna().any().any() or (values < 0).any().any() or (values % 1 != 0).any().any():
        raise ValueError("Counts must be nonnegative integers without missing values.")
    return values.astype("int64")


source = validate_counts(source_counts, ["soa", "dnskey", "nsec", "nsec3"])
www = validate_counts(www_counts, ["dnskey", "nsec", "nsec3"])
if source["soa"].eq(0).any() or source["dnskey"].eq(0).any() or www["dnskey"].eq(0).any():
    raise ValueError("Percentage denominators must be positive.")

# Keep machine-readable columns separate from the presentation format.
stats = source.add_prefix("source_").join(www.add_prefix("www_"))
stats["source_dnskey_pct"] = 100 * source["dnskey"] / source["soa"]
for mechanism in ("nsec", "nsec3"):
    stats[f"source_{mechanism}_pct"] = 100 * source[mechanism] / source["dnskey"]
    stats[f"www_{mechanism}_pct"] = 100 * www[mechanism] / www["dnskey"]

column_spec = [
    ("Source domains", "SOA #", "source_soa"),
    ("Source domains", "DNSKEY #", "source_dnskey"),
    ("Source domains", "DNSKEY %", "source_dnskey_pct"),
    ("Source domains", "NSEC #", "source_nsec"),
    ("Source domains", "NSEC %", "source_nsec_pct"),
    ("Source domains", "NSEC3 #", "source_nsec3"),
    ("Source domains", "NSEC3 %", "source_nsec3_pct"),
    ("WWW subdomains", "DNSKEY #", "www_dnskey"),
    ("WWW subdomains", "NSEC #", "www_nsec"),
    ("WWW subdomains", "NSEC %", "www_nsec_pct"),
    ("WWW subdomains", "NSEC3 #", "www_nsec3"),
    ("WWW subdomains", "NSEC3 %", "www_nsec3_pct"),
]
table1 = stats[[key for _, _, key in column_spec]].copy()
table1.columns = pd.MultiIndex.from_tuples([(group, label) for group, label, _ in column_spec])
table1.index = table1.index.map(LABELS)
table1.index.name = "Dataset"
formatted_table1 = table1.copy().astype(object)
for column in table1.columns:
    if column[1].endswith("%"):
        formatted_table1[column] = table1[column].map(lambda value: f"{value:.2f}%")
    else:
        formatted_table1[column] = table1[column].map(lambda value: f"{value:,.0f}")
display(formatted_table1)

Source domains                                                    \
                  SOA #   DNSKEY # DNSKEY %     NSEC #  NSEC %    NSEC3 #   
Dataset                                                                     
.com        149,021,866  8,364,107    5.61%  1,577,074  18.86%  6,720,037   
.net         11,156,231    707,851    6.34%    160,534  22.68%    543,279   
.org         10,972,417    707,896    6.45%    146,299  20.67%    556,602   
Umbrella        384,922     26,034    6.76%     16,251  62.42%      9,523   
Tranco          968,036     79,790    8.24%     49,080  61.51%     30,109   
Radar         1,137,219    102,067    8.98%     65,042  63.72%     36,246   
Majestic        946,890     75,863    8.01%     40,987  54.03%     34,296   

                 WWW subdomains                                        
         NSEC3 %       DNSKEY #     NSEC #  NSEC %    NSEC3 # NSEC3 %  
Dataset                                                                
.com      80.34%      8,364,107  1,463,427  17.50%  6,561,180  78.44%  
.net      76.75%        707,851    155,078  21.91%    532,883  75.28%  
.org      78.63%        707,896    140,760  19.88%    548,415  77.47%  
Umbrella  36.58%         20,731     13,004  62.73%      6,952  33.53%  
Tranco    37.74%         71,512     42,812  59.87%     27,054  37.83%  
Radar     35.51%         93,880     58,725  62.55%     33,327  35.50%  
Majestic  45.21%         69,995     37,372  53.39%     31,237  44.63%

## Export
`table1_stats.csv` contains the input counts and unrounded numeric percentages. `table1_formatted.csv` contains the displayed table with two header rows. Existing exports at these paths are replaced when this cell runs with `EXPORT = True`.

In [4]:
if EXPORT:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    stats.to_csv(OUTPUT_DIR / "table1_stats.csv", index_label="zone")
    print(f"Saved table1_stats.csv and table1_formatted.csv to {OUTPUT_DIR}")
else:
    print("Export disabled.")

Saved table1_stats.csv and table1_formatted.csv to /u/yz6me/dnsnsec26/output/jupyter-notebook/table1_results
